# Power consumption — how it works

This is the explanation and reference for `quicksat`'s power consumption. It covers what the two consumption files hold, how the loads follow the scenario's payload and platform modes, the margins, and where the model stops.

It is not a tutorial. The worked example is `sample/power/power_consumption.ipynb`. This notebook still runs, so that it cannot drift from the code. It reads `docs/power/data/`, its own copy of the inputs. The figures quoted below are true of that data.

The scenario names the modes, and power says what each draws (`docs/explanations/decisions.md`, entry 10). The power budget, `docs/power/power_budget_ref.ipynb`, sets the loads against the generation.

In [1]:
import os
from pathlib import Path

import numpy as np

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from docs/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import u
from quicksat.power.consumption import PowerConsumption, run_consumption
from quicksat.scenario.config import Scenario
from quicksat.scenario.names import SLEW
from quicksat.scenario.run import run_scenario

DATA = Path("docs") / "power" / "data"
GROUPS = ("payload", "platform")
CONSUMPTION_FILES = {group: DATA / f"{group}_consumption.csv" for group in GROUPS}

consumption = {
    group: PowerConsumption.from_csv(path) for group, path in CONSUMPTION_FILES.items()
}
run = run_scenario(Scenario.from_yaml_file(DATA / "scenario.yaml"))
loads = run_consumption(consumption, run)

## The consumption files

The spacecraft has two modes at any time: the payload's, and the mode of the platform that carries it. So there are two consumption files, `payload_consumption.csv` and `platform_consumption.csv`. Each is a CSV, written by hand. No notebook writes them. Each has one row for each piece of equipment of its group that draws power, so it can be a long file. They are kept apart from the power system file for that reason. Equipment that draws none, such as the structure, is left out. The rows need not match the mass budget's equipment list.

Which equipment goes in which file is the user's choice. This notebook's files put the mass memory and the X-band transmitter with the payload, as ESA's payload data handling and transmission (PDHT) does. So the payload's `downlink` mode is the data dump, and the platform's `ttc` mode is the TT&C pass, with the S-band transmitter on and the wheels tracking the station.

| Column | What it holds |
| --- | --- |
| `equipment_id` | A short name with no spaces, which no other row of the file has |
| `equipment_name` | Free text |
| `subsystem` | Such as `ADCS`, with no spaces. The plots stack the loads by it. A subsystem may appear in both files, such as `Thermal`. |
| `units_on` | How many units draw power: 1 of a cold-redundant pair, or all four reaction wheels |
| `eqpt_margin` | The equipment's margin, in percent, such as `5` |
| one column for each of the group's modes | The power of one unit in that mode, such as `10 W`, averaged over the mode |
| `comments` | Optional free text |

A value is an average over the mode. So a heater that a thermostat switches, or a magnetorquer that runs now and then, gives its mean power. Every cell of a mode column needs a power, `0 W` included, so that no mode is ever assumed to draw what another does.

This notebook's files:

In [2]:
for path in CONSUMPTION_FILES.values():
    print(f"{path.name}:\n{path.read_text()}")

payload_consumption.csv:
equipment_id,equipment_name,subsystem,units_on,eqpt_margin,idle,imaging,downlink,comments,thermal
x_band_tx,X-band transmitter,COMM,1,10,0 W,0 W,55 W,About 5-10 W of radio output,inside_mli
mass_memory,SSDR 2 TB,OBDH,1,10,10 W,30 W,25 W,"Standby, recording, playback",inside_mli
payload_electronics,Video processing unit,Payload,1,10,5 W,40 W,5 W,Compresses the images,inside_mli
focal_plane,Focal plane assembly,Payload,1,20,0 W,45 W,0 W,"Detectors, their electronics and coolers; new design",inside_mli
payload_heaters,Payload heater lines,Thermal,1,20,22.1 W,0 W,0 W,"Thermostat mean in idle only, to balance the run; 37.6 W installed",heater

platform_consumption.csv:
equipment_id,equipment_name,subsystem,units_on,eqpt_margin,idle,imaging,ttc,slew,comments
obc,SSTL OBC750,OBDH,1,5,10 W,10 W,10 W,10 W,Cold redundant: one of two on
star_tracker,Jena Astro HP optical head,ADCS,2,5,6 W,6 W,6 W,6 W,Both heads on
reaction_wheel,Rockwell RSI 45,ADCS,4,5,10 W,12 W,15 W,10 

## The modes, slews included

At each time of the run, each row draws its units' power in its group's mode of that time. The scenario names both modes in each activity, such as `{payload_mode: imaging, platform_mode: imaging}`. Platform equipment that changes with the payload's work takes a platform mode of its own: here the platform's `imaging` parks the solar wings and holds the wheels steadier.

While the body turns between attitudes, the run gives the platform the mode `slew` instead (see "Slews" in `docs/scenario/scenario_ref.ipynb`). So a scenario with slews needs a `slew` column in the platform's file, which only the user can fill. The payload's mode carries on through the turn, since its work, such as a file transfer, does not depend on the attitude. So the payload's file needs no `slew` column.

Every mode the run gives a group needs its column in that group's file, or the consumption stops with the modes it lacks:

In [3]:
path = CONSUMPTION_FILES["platform"]
renamed = DATA.parent / "platform_renamed.csv"
renamed.write_text(path.read_text().replace(",slew,", ",turning,"))
try:
    run_consumption(consumption | {"platform": PowerConsumption.from_csv(renamed)}, run)
except ValueError as error:
    print(error)
finally:
    renamed.unlink()

platform_renamed.csv has no column for the platform's modes ['slew'], which the run has. It has columns for ['idle', 'imaging', 'ttc', 'turning'].


A column for a mode the run does not have, such as `safe`, is allowed. The run's `notes` name it, since it is not used. The run is cut wherever either group's mode changes, and each piece holds both modes. This notebook's run has every mode of both files:

In [4]:
for group in GROUPS:
    print(group, sorted({modes[group] for _, _, modes in loads.pieces}))
print(loads.notes or "no notes")

payload ['downlink', 'idle', 'imaging']
platform ['idle', 'imaging', 'slew', 'ttc']
no notes


## Margins

Each row carries its own margin, as in the mass budget: 5% for equipment that has flown, 10% for equipment that needs changes, and 20% for new designs, such as the focal plane, and for the heaters, whose power depends on the thermal design. The power budget adds a system margin on all the loads together, from the power system file.

Each group's equipment together, in W. At any time, the spacecraft draws its payload mode's total and its platform mode's together:

In [5]:
for group, loads_file in consumption.items():
    display(loads_file.mode_totals().style.format("{:.1f}").set_caption(f"The {group}"))

,idle [W],imaging [W],downlink [W]
without margins,37.1,115.0,85.0
with equipment margins,43.0,131.0,93.5


,idle [W],imaging [W],ttc [W],slew [W]
without margins,138.0,140.0,173.0,138.0
with equipment margins,149.1,150.9,185.8,149.1


## The loads over a run

The modes change at exact times, between the steps of the grid. So the energy is worked out from the mode intervals, not from the grid: within each piece, where neither group's mode changes, the power holds steady. `energy_until` gives it at any time, exactly.

`orbit_table` gives each whole orbit, counted from the start of the run as for the generation: the mean and the peak load, the energy drawn, and the part drawn in eclipse. The battery must give that part, through its discharge path. Here the eclipse falls in nadir idle:

In [6]:
loads.orbit_table.style.hide(axis="index").format(
    {
        "mean load [W]": "{:.1f}",
        "peak load [W]": "{:.1f}",
        "energy [Wh]": "{:.1f}",
        "in eclipse [Wh]": "{:.1f}",
    }
)

orbit,start [UTC],mean load [W],peak load [W],energy [Wh],in eclipse [Wh]
1,2026-10-01T09:00:00.000,208.3,281.9,329.3,109.9
2,2026-10-01T10:34:51.055,207.6,281.9,328.2,109.9
3,2026-10-01T12:09:42.109,207.0,281.9,327.3,109.9


The energy over the whole run, from the grid and from the mode intervals. On the grid, each step takes the mode at its start, so it misses where a mode starts or ends between two steps:

In [7]:
seconds = np.asarray((run.times - run.times[0]).to_value(u.s))
exact = loads.energy_until(np.array([seconds[-1]]))[0] / 3600
on_grid = (
    float(np.sum(np.asarray(loads.total.to_value(u.W))[:-1] * np.diff(seconds))) / 3600
)
slewing = sum(
    end - start for start, end, modes in loads.pieces if modes["platform"] == SLEW
)
print(f"from the mode intervals  {exact:.2f} W h")
print(f"from the grid            {on_grid:.2f} W h")
print(f"time in slews            {slewing / 60:.1f} min")

from the mode intervals  984.76 W h
from the grid            985.11 W h
time in slews            27.0 min


## Limitations

What the consumption does not do, yet or at all:

- **Averages, not peaks.** Each mode gives a mean power. Inrush currents, a heater's on and off, or a wheel's spin-up are not followed.
- **One power for each mode.** The power does not depend on the attitude, the illumination or the temperature. Heaters draw the same in sunlight as in eclipse, unless the activities use modes of their own for them.
- **Two modes at a time, set together.** Each activity sets the payload's mode and the platform's. Subsystems within a group do not change on their own: a TT&C pass during imaging needs an activity of its own, or a platform mode that covers both.
- **No safe mode by itself.** A safe mode is a mode like any other, with its own columns and activities.